# Task 1.1 — Bronze Ingestion: LMS Events + Video Watch

## Notebook: 01_bronze_lms_video_ingestion

**Pipeline:** EduPath Learning Analytics · Medallion Architecture

**Layer:** Bronze (raw ingestion, no transformations)


**What this notebook does:**
- Ingests `lms_events.json` (NDJSON) using Auto Loader (cloudFiles) → `bronze.lms_events_bronze`
- Ingests `video_watch_events.parquet` using batch Parquet read → `bronze.video_watch_bronze`
- Adds metadata columns: `_source_file`, `_load_ts`, `_schema_version`, partition column
- Runs DQ checks: future timestamps, NULL student_id, volume anomaly alerts


**Business Rules enforced:**
- `event_ts` must NOT be future-dated
- `student_id` must NOT be NULL in any row
- Daily LMS event count must be between 1M and 10M (volume anomaly alert)

**Run order:** Run all cells top-to-bottom. Safe to re-run (idempotent).

In [0]:
# ============================================================
# CELL 2 — Catalog config and all path/table name constants
# Run this cell first every time before executing any other cell
# ============================================================

# --- Catalog & schema ---
CATALOG  = "mini_project_grp6"
BRONZE   = "bronze"

# --- Volume source paths (files you uploaded in setup) ---
LMS_RAW_PATH         = f"/Volumes/{CATALOG}/{BRONZE}/lms_events_raw/"
VIDEO_RAW_PATH       = f"/Volumes/{CATALOG}/{BRONZE}/video_watch_raw/"

# --- Auto Loader checkpoint paths (one sub-folder per stream) ---
LMS_CHECKPOINT_PATH  = f"/Volumes/{CATALOG}/{BRONZE}/checkpoints/lms/"
VIDEO_CHECKPOINT_PATH= f"/Volumes/{CATALOG}/{BRONZE}/checkpoints/video/"

# --- Target Delta table names (fully qualified) ---
LMS_BRONZE_TABLE     = f"{CATALOG}.{BRONZE}.lms_events_bronze"
VIDEO_BRONZE_TABLE   = f"{CATALOG}.{BRONZE}.video_watch_bronze"
DQ_AUDIT_TABLE       = f"{CATALOG}.audit.dq_audit"

# --- Schema version tag (bump when source schema changes) ---
SCHEMA_VERSION       = "v1.0"

# --- Volume anomaly thresholds for LMS events ---
LMS_MIN_DAILY_COUNT  = 1_000_000
LMS_MAX_DAILY_COUNT  = 10_000_000

# --- Set default catalog + schema for this session ---
spark.catalog.setCurrentCatalog(CATALOG)
spark.catalog.setCurrentDatabase(BRONZE)

print(f"✅ Catalog  : {CATALOG}")
print(f"✅ Schema   : {BRONZE}")
print(f"✅ LMS path : {LMS_RAW_PATH}")
print(f"✅ Video path: {VIDEO_RAW_PATH}")

In [0]:
# ============================================================
# CELL 3 — All imports needed for this notebook
# ============================================================

from pyspark.sql import functions as F
from pyspark.sql.types import (
    StructType, StructField,
    StringType, LongType, TimestampType, IntegerType, DoubleType
)
from delta.tables import DeltaTable
from datetime import datetime, timezone

print("✅ Imports successful")


## Part A — LMS Events Bronze (`lms_events_bronze`)

**Source:** `lms_events.json` (NDJSON) from Volume `lms_events_raw/`

**Method:** Auto Loader (`cloudFiles`) — detects new files automatically, maintains checkpoint


Schema inference on first run, schema evolution handled via `mergeSchema`

Partitioned by `event_date` (derived from `event_ts`)

Adds: `_source_file`, `_load_ts`, `_schema_version`

In [0]:
# ============================================================
# CELL 5 — Explicit schema for lms_events NDJSON
# Defining schema explicitly is best practice for Auto Loader
# to avoid schema inference cost on every run
# ============================================================

lms_schema = StructType([
    StructField("event_id",         StringType(),    nullable=False),
    StructField("student_id",       StringType(),    nullable=True),   # nullable=True so we can catch NULLs in DQ
    StructField("course_id",        StringType(),    nullable=True),
    StructField("module_id",        StringType(),    nullable=True),
    StructField("event_type",       StringType(),    nullable=True),
    StructField("event_ts",         TimestampType(), nullable=True),
    StructField("duration_seconds", LongType(),      nullable=True),
    StructField("page_url",         StringType(),    nullable=True),
    StructField("device_type",      StringType(),    nullable=True),
    StructField("ip_country",       StringType(),    nullable=True),
])

print("✅ LMS schema defined with", len(lms_schema.fields), "fields")

In [0]:
# ============================================================
# CELL 6 — Auto Loader: read LMS NDJSON → add metadata → write Bronze
#
# Auto Loader options explained:
#   cloudFiles.format       = source file format (json)
#   cloudFiles.schemaLocation = where Auto Loader stores inferred schema
#   cloudFiles.inferColumnTypes = cast types automatically
#   multiLine = False       = NDJSON (one JSON object per line)
#   mergeSchema             = allow new columns without breaking pipeline
# ============================================================

lms_stream = (
    spark.readStream
    .format("cloudFiles")
    .option("cloudFiles.format",         "json")
    .option("cloudFiles.schemaLocation",  f"{LMS_CHECKPOINT_PATH}schema/")
    .option("cloudFiles.inferColumnTypes", "true")
    .option("multiLine",                  "false")           # NDJSON = one record per line
    .schema(lms_schema)                                        # use explicit schema
    .load(LMS_RAW_PATH)
    # ── Add Bronze metadata columns ──────────────────────────
    .withColumn("_source_file",    F.col("_metadata.file_path"))
    .withColumn("_load_ts",        F.current_timestamp())
    .withColumn("_schema_version", F.lit(SCHEMA_VERSION))
    # ── Derive partition column from event_ts ─────────────────
    .withColumn("event_date",      F.to_date(F.col("event_ts")))
)

# Write streaming query to Delta table, partitioned by event_date
lms_write_query = (
    lms_stream.writeStream
    .format("delta")
    .outputMode("append")
    .option("checkpointLocation",  LMS_CHECKPOINT_PATH)
    .option("mergeSchema",          "true")
    .partitionBy("event_date")
    .trigger(availableNow=True)              # process all available files then stop (batch mode)
    .toTable(LMS_BRONZE_TABLE)
)

# Wait for the streaming batch to complete before moving to next cell
lms_write_query.awaitTermination()
print(f"✅ LMS events written to: {LMS_BRONZE_TABLE}")

In [0]:
# ============================================================
# CELL 7 — Quick verification of lms_events_bronze
# ============================================================

lms_df = spark.table(LMS_BRONZE_TABLE)

print("── lms_events_bronze ─────────────────────────────")
print(f"Total rows   : {lms_df.count():,}")
print(f"Columns      : {len(lms_df.columns)}")
print(f"Partitions   : event_date")
print()

# Show distinct event types
print("── event_type distribution ───────────────────────")
lms_df.groupBy("event_type").count().orderBy(F.desc("count")).show(truncate=False)

# Show metadata columns
print("── metadata columns sample ───────────────────────")
lms_df.select("event_id", "_source_file", "_load_ts", "_schema_version", "event_date").show(3, truncate=60)


## Part B — Video Watch Events Bronze (`video_watch_bronze`)

**Source:** `video_watch_events.parquet` from Volume `video_watch_raw/`

**Method:** Batch Parquet read with explicit schema validation


Parquet preserves column types natively — no casting needed

Partitioned by `watch_date` (derived from `watch_date` field)

Adds: `_source_file`, `_load_ts`, `_schema_version`

In [0]:
# ============================================================
# CELL 9 — Explicit schema for video_watch_events Parquet
# Even though Parquet carries its own schema, we define an
# expected schema to catch upstream schema drift early
# ============================================================

video_schema = StructType([
    StructField("watch_id",               StringType(),    nullable=False),
    StructField("student_id",             StringType(),    nullable=True),
    StructField("course_id",              StringType(),    nullable=True),
    StructField("video_id",               StringType(),    nullable=True),
    StructField("video_title",            StringType(),    nullable=True),
    StructField("video_duration_seconds", LongType(),      nullable=True),
    StructField("watched_seconds",        LongType(),      nullable=True),
    StructField("completion_pct",         DoubleType(),    nullable=True),
    StructField("play_count",             IntegerType(),   nullable=True),
    StructField("pause_count",            IntegerType(),   nullable=True),
    StructField("seek_count",             IntegerType(),   nullable=True),
    StructField("playback_speed",         DoubleType(),    nullable=True),
    StructField("watch_date",             StringType(),    nullable=True),   # cast to date below
    StructField("device_type",            StringType(),    nullable=True),
    StructField("buffering_events",       IntegerType(),   nullable=True),
])

print("✅ Video schema defined with", len(video_schema.fields), "fields")

In [0]:
# ============================================================
# CELL 10 — Read Parquet files with schema validation
# We do a two-step approach:
#   1. Read raw Parquet (let Spark infer from file)
#   2. Validate that all expected columns exist
#   3. Cast watch_date properly
# ============================================================

# Step 1: Read all Parquet files in the volume folder
video_raw = (
    spark.read
    .format("parquet")
    .option("mergeSchema", "true")     # handle multiple Parquet files with slight schema differences
    .load(VIDEO_RAW_PATH)
)

# Step 2: Validate expected columns exist in the file
expected_cols = set([f.name for f in video_schema.fields])
actual_cols   = set(video_raw.columns)
missing_cols  = expected_cols - actual_cols

if missing_cols:
    raise ValueError(f"❌ Schema validation failed — missing columns: {missing_cols}")
else:
    print(f"✅ Schema validation passed — all {len(expected_cols)} expected columns present")
    print(f"   Total columns in file: {len(actual_cols)}")

In [0]:
# ============================================================
# CELL 11 — Add metadata, cast watch_date, write to Delta
# ============================================================

video_bronze_df = (
    video_raw
    # ── Cast watch_date string → proper DateType ──────────────
    .withColumn("watch_date",      F.to_date(F.col("watch_date")))
    # ── Add Bronze metadata columns ───────────────────────────
    .withColumn("_source_file",    F.col("_metadata.file_path"))
    .withColumn("_load_ts",        F.current_timestamp())
    .withColumn("_schema_version", F.lit(SCHEMA_VERSION))
)

# Write to Delta table — APPEND mode, partition by watch_date
(
    video_bronze_df.write
    .format("delta")
    .mode("append")
    .option("mergeSchema", "true")
    .partitionBy("watch_date")
    .saveAsTable(VIDEO_BRONZE_TABLE)
)

print(f"✅ Video watch events written to: {VIDEO_BRONZE_TABLE}")
print(f"   Rows written: {video_bronze_df.count():,}")

In [0]:
# ============================================================
# CELL 12 — Verify video_watch_bronze
# ============================================================

video_df = spark.table(VIDEO_BRONZE_TABLE)

print("── video_watch_bronze ────────────────────────────")
print(f"Total rows   : {video_df.count():,}")
print(f"Columns      : {len(video_df.columns)}")
print()

# Completion pct stats
print("── completion_pct stats ──────────────────────────")
video_df.select(
    F.min("completion_pct").alias("min_pct"),
    F.max("completion_pct").alias("max_pct"),
    F.round(F.avg("completion_pct"), 2).alias("avg_pct")
).show()

print("── metadata sample ───────────────────────────────")
video_df.select("watch_id", "watch_date", "_load_ts", "_schema_version").show(3, truncate=50)

In [0]:
# ============================================================
# CELL 13 — DQ Check 1: event_ts must not be future-dated
# Business rule from Task 1.1 spec
# Rows that fail are written to audit.dq_audit, NOT dropped from bronze
# ============================================================

now_ts = F.current_timestamp()

# Find rows where event_ts is in the future
future_events = (
    spark.table(LMS_BRONZE_TABLE)
    .filter(F.col("event_ts") > now_ts)
    .withColumn("dq_check_name",   F.lit("future_event_ts"))
    .withColumn("dq_table",        F.lit(LMS_BRONZE_TABLE))
    .withColumn("dq_severity",     F.lit("HIGH"))
    .withColumn("dq_checked_at",   F.current_timestamp())
    .withColumn("dq_message",      F.concat_ws(" | ",
        F.lit("event_ts is future-dated:"),
        F.col("event_ts").cast("string"),
        F.lit("for event_id:"),
        F.col("event_id")
    ))
    .select("dq_check_name", "dq_table", "dq_severity", "dq_checked_at", "dq_message",
            "event_id", "student_id", "event_ts")
)

future_count = future_events.count()
print(f"── DQ Check 1: future-dated event_ts ─────────────")
print(f"   Rows with future event_ts: {future_count}")

if future_count > 0:
    # Write flagged rows to audit table
    (
        future_events.write
        .format("delta")
        .mode("append")
        .option("mergeSchema", "true")
        .saveAsTable(DQ_AUDIT_TABLE)
    )
    print(f"   ⚠ {future_count} rows flagged and written to {DQ_AUDIT_TABLE}")
else:
    print("   ✅ PASSED — no future-dated events found")

In [0]:
# ============================================================
# CELL 14 — DQ Check 2: student_id must not be NULL
# Runs on both lms_events_bronze and video_watch_bronze
# Also tracks completeness metric (target: < 0.5% missing)
# ============================================================

def check_null_student_id(table_name: str) -> None:
    df        = spark.table(table_name)
    total     = df.count()
    null_rows = df.filter(F.col("student_id").isNull())
    null_count= null_rows.count()
    pct       = (null_count / total * 100) if total > 0 else 0

    print(f"── DQ Check 2: NULL student_id in {table_name.split('.')[-1]} ──")
    print(f"   Total rows : {total:,}")
    print(f"   NULL count : {null_count:,}  ({pct:.3f}%)")

    if null_count > 0:
        # Flag rows and write to audit
        flagged = (
            null_rows
            .withColumn("dq_check_name",  F.lit("null_student_id"))
            .withColumn("dq_table",       F.lit(table_name))
            .withColumn("dq_severity",    F.lit("HIGH"))
            .withColumn("dq_checked_at",  F.current_timestamp())
            .withColumn("dq_message",     F.lit("student_id is NULL"))
        )
        flagged.select(
            "dq_check_name", "dq_table", "dq_severity", "dq_checked_at", "dq_message"
        ).write.format("delta").mode("append").option("mergeSchema","true").saveAsTable(DQ_AUDIT_TABLE)
        print(f"   ⚠ FLAGGED — {null_count} rows written to dq_audit")
    else:
        print("   ✅ PASSED")

    # Alert if completeness exceeds 0.5% threshold
    if pct > 0.5:
        print(f"   🚨 ALERT — NULL rate {pct:.2f}% exceeds 0.5% threshold!")
    print()

check_null_student_id(LMS_BRONZE_TABLE)
check_null_student_id(VIDEO_BRONZE_TABLE)

In [0]:
# ============================================================
# CELL 15 — DQ Check 3: LMS daily event volume anomaly
# Alert if daily count < 1M or > 10M
# In Airflow this becomes a Slack alert; here we print a warning
# In production: replace print with a Slack webhook call
# ============================================================

lms_daily_counts = (
    spark.table(LMS_BRONZE_TABLE)
    .groupBy("event_date")
    .agg(F.count("*").alias("daily_count"))
    .orderBy("event_date")
)

print("── DQ Check 3: LMS daily volume anomaly ──────────")
print(f"   Alert thresholds: MIN={LMS_MIN_DAILY_COUNT:,}  MAX={LMS_MAX_DAILY_COUNT:,}")
print()

# Collect to driver (small result — one row per date)
daily_rows = lms_daily_counts.collect()

anomalies = 0
for row in daily_rows:
    status = "✅"
    alert_msg = ""
    if row["daily_count"] < LMS_MIN_DAILY_COUNT:
        status    = "⚠ LOW"
        alert_msg = f"  ← ANOMALY: below {LMS_MIN_DAILY_COUNT:,} threshold"
        anomalies += 1
    elif row["daily_count"] > LMS_MAX_DAILY_COUNT:
        status    = "⚠ HIGH"
        alert_msg = f"  ← ANOMALY: above {LMS_MAX_DAILY_COUNT:,} threshold"
        anomalies += 1
    print(f"   {status}  {row['event_date']}  count={row['daily_count']:,}{alert_msg}")

print()
if anomalies == 0:
    print("   ✅ PASSED — all daily counts within expected range")
else:
    print(f"   🚨 {anomalies} date(s) outside thresholds — would trigger Slack alert in production")
    # Production hook: call Slack webhook here
    # import requests
    # requests.post(SLACK_WEBHOOK_URL, json={"text": f"Volume anomaly: {anomalies} dates flagged"})

In [0]:
%sql
-- ============================================================
-- CELL 16 — SQL verification of both Bronze tables
-- ============================================================

-- 1. Row counts and partition column check
SELECT
    'lms_events_bronze'   AS table_name,
    COUNT(*)               AS total_rows,
    COUNT(DISTINCT event_date) AS distinct_dates,
    MIN(event_date)         AS earliest_date,
    MAX(event_date)         AS latest_date,
    COUNT(DISTINCT student_id) AS unique_students
FROM mini_project_grp6.bronze.lms_events_bronze

UNION ALL

SELECT
    'video_watch_bronze'  AS table_name,
    COUNT(*)               AS total_rows,
    COUNT(DISTINCT watch_date) AS distinct_dates,
    MIN(watch_date)         AS earliest_date,
    MAX(watch_date)         AS latest_date,
    COUNT(DISTINCT student_id) AS unique_students
FROM mini_project_grp6.bronze.video_watch_bronze;

In [0]:
# ============================================================
# CELL 17 — Task 1.1 completion summary
# ============================================================

lms_count   = spark.table(LMS_BRONZE_TABLE).count()
video_count = spark.table(VIDEO_BRONZE_TABLE).count()

print("═" * 55)
print("  TASK 1.1 COMPLETE — Bronze LMS + Video Ingestion")
print("═" * 55)
print()
print(f"  ✅ {LMS_BRONZE_TABLE}")
print(f"      Rows      : {lms_count:,}")
print(f"      Partition : event_date")
print(f"      Metadata  : _source_file · _load_ts · _schema_version")
print()
print(f"  ✅ {VIDEO_BRONZE_TABLE}")
print(f"      Rows      : {video_count:,}")
print(f"      Partition : watch_date")
print(f"      Metadata  : _source_file · _load_ts · _schema_version")
print()
print("  DQ checks run:")
print("      Cell 13 — future-dated event_ts")
print("      Cell 14 — NULL student_id (both tables)")
print("      Cell 15 — LMS daily volume anomaly")
print()
print("  ─────────────────────────────────────────────────")
print("  Next: Task 1.2 → 02_bronze_quiz_enrollments")
print("         Quiz CSV incremental + Enrollments MERGE INTO")
print("═" * 55)